# 10 — AI-agenter: Verktøy, minne og ReAct

**Fase:** 2 — Kjerne AI | **Tid:** 2 timer | **Krav:** Notatbok 05, 08

**Hva du bygger:** En pensjonschatbot som ikke bare svarer — den *bestemmer* hva den trenger, kaller verktøy og setter svar sammen fra flere kilder.

---

## RAG vs. Agent

```
RAG:    Spørsmål → Hent → Svar          (én tur, passiv)
Agent:  Spørsmål → Tenk → Handling
                        → Tenk → Handling
                        → Tenk → Svar   (flere turer, aktiv)
```

En agent kan:
- Bruke verktøy (søk, kalkulator, API-kall, database)
- Huske hva den har gjort i samtalen
- Ta beslutninger om hva neste steg er
- Avbryte og prøve noe annet hvis det ikke funker

**ReAct-mønsteret** (Reason + Act) er den enkleste agent-arkitekturen:
```
Tenk: "Jeg trenger å slå opp AFP-regler"
Handling: kall søk_dokument("AFP")
Observer: [resultat fra søk]
Tenk: "Nå har jeg nok info til å svare"
Svar: "AFP gir deg..."
```

In [ ]:
%pip install -q openai sentence-transformers chromadb

In [ ]:
import json
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb

llm  = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODELL = "llama3.2"
embed = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Sett opp en liten vektordatabase for pensjonsdokumenter
db = chromadb.PersistentClient(path="./agent_db")
kol = db.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})

docs = [
    ("afp-1", "AFP kan tas ut fra 62 år for offentlig ansatte med minst 3 års tjeneste."),
    ("afp-2", "AFP er livsvarig og kombineres med alderspensjon fra 67 år."),
    ("ald-1", "Alderspensjon fra SPK utbetales fra 67 år, full pensjon krever 30 opptjeningsår."),
    ("ufø-1", "Uførepensjon innvilges ved minst 20% varig nedsatt arbeidsevne, sats 66%."),
]
if kol.count() == 0:
    kol.add(
        ids=[d[0] for d in docs],
        documents=[d[1] for d in docs],
        embeddings=embed.encode([d[1] for d in docs]).tolist(),
    )
print("Klar.")

---

## Del 1: Definer verktøy

In [ ]:
# --- Verktøyfunksjoner (det agenten kan "gjøre") ---

def søk_pensjonsdokument(spørsmål: str) -> str:
    """Søk i pensjonsdokumenter og returner relevante biter."""
    sv = embed.encode([spørsmål]).tolist()
    res = kol.query(query_embeddings=sv, n_results=2)
    return "\n".join(res["documents"][0])

def beregn_pensjon(alder: int, opptjeningsår: int, lønn: int) -> str:
    """Enkel pensjonsberegning (forenklet)."""
    if opptjeningsår >= 30:
        sats = 0.66
    else:
        sats = 0.66 * (opptjeningsår / 30)
    månedlig = int(lønn * sats / 12)
    return f"Estimert månedlig pensjon: {månedlig:,} kr (sats: {sats:.0%}, basert på {opptjeningsår} opptjeningsår)"

# --- Verktøydefinisjoner for LLM-API ---
VERKTØY = [
    {
        "type": "function",
        "function": {
            "name": "søk_pensjonsdokument",
            "description": "Søk etter informasjon om pensjonsregler og rettigheter",
            "parameters": {
                "type": "object",
                "properties": {
                    "spørsmål": {"type": "string", "description": "Søkespørsmål"}
                },
                "required": ["spørsmål"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "beregn_pensjon",
            "description": "Beregn estimert månedlig pensjon basert på alder, opptjeningsår og lønn",
            "parameters": {
                "type": "object",
                "properties": {
                    "alder":         {"type": "integer", "description": "Alder i år"},
                    "opptjeningsår": {"type": "integer", "description": "År i pensjonssystemet"},
                    "lønn":          {"type": "integer", "description": "Årslønn i kroner"}
                },
                "required": ["alder", "opptjeningsår", "lønn"]
            }
        }
    }
]

VERKTØY_MAP = {
    "søk_pensjonsdokument": søk_pensjonsdokument,
    "beregn_pensjon": beregn_pensjon,
}

print("Verktøy definert:", list(VERKTØY_MAP.keys()))

---

## Del 2: ReAct-agentløkke

In [ ]:
def kjør_agent(bruker_spørsmål: str, maks_turer: int = 5) -> str:
    """
    ReAct-loop:
    1. LLM bestemmer seg for et verktøy
    2. Vi kjører verktøyet
    3. Resultatet sendes tilbake til LLM
    4. Gjenta til LLM svarer uten verktøykall
    """
    meldinger = [
        {
            "role": "system",
            "content": (
                "Du er en pensjonsrådgiver hos SPK. "
                "Bruk søk_pensjonsdokument for regler, beregn_pensjon for estimater. "
                "Svar alltid på norsk."
            )
        },
        {"role": "user", "content": bruker_spørsmål}
    ]

    for tur in range(maks_turer):
        svar = llm.chat.completions.create(
            model=MODELL,
            messages=meldinger,
            tools=VERKTØY,
            tool_choice="auto",
        )
        melding = svar.choices[0].message

        if not melding.tool_calls:
            # Ingen verktøykall → agenten er ferdig
            print(f"  [Ferdig etter {tur+1} tur(er)]")
            return melding.content

        # Kjør hvert verktøykall
        meldinger.append(melding)
        for kall in melding.tool_calls:
            args      = json.loads(kall.function.arguments)
            funksjon  = VERKTØY_MAP[kall.function.name]
            resultat  = funksjon(**args)
            print(f"  → Kaller {kall.function.name}({args})")
            print(f"  ← {resultat[:100]}..." if len(resultat) > 100 else f"  ← {resultat}")

            meldinger.append({
                "role": "tool",
                "tool_call_id": kall.id,
                "content": resultat,
            })

    return "Nådde maks antall turer."


# Test agenten
print("=" * 50)
print("Q: Kan jeg gå av med AFP, og hva vil jeg få?")
print("=" * 50)
svar = kjør_agent("Jeg er 58 år, har jobbet i staten i 25 år og tjener 650 000 kr. Kan jeg gå av med AFP, og hva vil jeg ca få?")
print(f"\nSvar:\n{svar}")

---

## Del 3: De tre minnetypene

Agenter trenger minne for å fungere over tid:

In [ ]:
# Type 1: In-context minne (meldingshistorikk i prompt)
# → Mister alt når kontekstvindyet er fullt
samtale_historikk = []  # Akkumuleres i prompt

# Type 2: Eksternt minne (vektordatabase)
# → Vedvarende, skalerbart
# → Allerede brukt over (ChromaDB)

# Type 3: Episodisk minne (oppsummert samtaledatabase)
import json, pathlib

MINNE_FIL = pathlib.Path("./agent_minne.json")

def lagre_i_minne(bruker: str, nøkkelinfo: str):
    """Lagre viktig info om brukeren for fremtidige samtaler."""
    minne = json.loads(MINNE_FIL.read_text()) if MINNE_FIL.exists() else {}
    minne[bruker] = nøkkelinfo
    MINNE_FIL.write_text(json.dumps(minne, ensure_ascii=False, indent=2))

def hent_fra_minne(bruker: str) -> str:
    if not MINNE_FIL.exists():
        return "Ingen tidligere info."
    minne = json.loads(MINNE_FIL.read_text())
    return minne.get(bruker, "Ingen tidligere info.")

# Lagre brukerinfo
lagre_i_minne("bruker_123", "Alder: 58, opptjeningsår: 25, lønn: 650000, interessert i AFP")

# Neste samtale — agenten husker deg
print("Henter minne:", hent_fra_minne("bruker_123"))

---

## Oppsummering

| Komponent | Hva det gjør |
|-----------|-------------|
| Verktøy | Funksjoner agenten kan kalle |
| ReAct-løkke | Tenk→Handling→Observer til ferdig |
| In-context minne | Meldingshistorikk i prompt |
| Eksternt minne | Vektordatabase — skalerbart |
| Episodisk minne | Lagret oppsummering mellom samtaler |

---

## Hva er neste steg?

**Neste: `11_agent_frameworks.ipynb`** — Du har bygget en agent fra bunnen. Nå ser vi på rammeverk (LangGraph, CrewAI) som gjør det enklere å bygge komplekse agentsystemer.